In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Reproducibility configuration
SEEDS = range(2026, 2036)
EMBEDDING_DIM = 16
REFERENCE_SAMPLES = 500
CALIBRATION_WINDOWS = 2000
TEST_WINDOWS = 1000
VALIDATION_WINDOWS = 5000
WINDOW_SIZE = 20
TARGET_FALSE_ALARM_RATE = 0.01

MEAN_SHIFT_LEVELS = [0.00, 0.02, 0.05, 0.10, 0.15]
VARIANCE_SCALE_LEVELS = [1.00, 1.10, 1.25, 1.50, 2.00]

print("Configuration loaded.")
print("Seeds:", list(SEEDS))
print("Embedding dimensions:", EMBEDDING_DIM)
print("Calibration windows:", CALIBRATION_WINDOWS)
print("Test windows per scenario:", TEST_WINDOWS)
print("Validation windows:", VALIDATION_WINDOWS)


Configuration loaded.
Seeds: [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035]
Embedding dimensions: 16
Calibration windows: 2000
Test windows per scenario: 1000
Validation windows: 5000


In [ ]:

def euclidean_score(reference_mean, window):
    """Distance between reference mean and current-window mean."""
    window_mean = np.mean(window, axis=0)
    return np.linalg.norm(window_mean - reference_mean)


def mahalanobis_score(reference_mean, reference_covariance, window):
    """Covariance-normalized distance between window and reference means."""
    window_mean = np.mean(window, axis=0)
    delta = window_mean - reference_mean

    regularized_covariance = (
        reference_covariance
        + 1e-6 * np.eye(len(reference_mean))
    )

    inverse_covariance = np.linalg.pinv(regularized_covariance)
    squared_distance = delta @ inverse_covariance @ delta

    return float(np.sqrt(max(squared_distance, 0.0)))



def variance_score(reference_data, current_window):
    """Ratio of current-window spread to reference-data spread."""

    reference_spread = np.mean(
        np.sum(
            (reference_data - reference_data.mean(axis=0)) ** 2,
            axis=1
        )
    )

    current_spread = np.mean(
        np.sum(
            (current_window - current_window.mean(axis=0)) ** 2,
            axis=1
        )
    )

    return float(current_spread / reference_spread)



print("Three detector functions defined successfully.")


Three detector functions defined successfully.


In [ ]:

def generate_reference(rng):
    """Generate synthetic reference embeddings."""
    reference = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(REFERENCE_SAMPLES, EMBEDDING_DIM)
    )
    reference_mean = reference.mean(axis=0)
    reference_covariance = np.cov(reference, rowvar=False)
    reference_variance = reference.var(axis=0, ddof=1)

    return reference, reference_mean, reference_covariance, reference_variance


def generate_window(rng, mean_shift=0.0, variance_scale=1.0):
    """Generate one synthetic Gaussian embedding window."""
    return rng.normal(
        loc=mean_shift,
        scale=np.sqrt(variance_scale),
        size=(WINDOW_SIZE, EMBEDDING_DIM)
    )



def calculate_scores(reference_mean, reference_covariance,
                     reference_data, window):
    """Calculate all three detector scores for a window."""
    return {
        "euclidean": euclidean_score(reference_mean, window),
        "mahalanobis": mahalanobis_score(
            reference_mean, reference_covariance, window
        ),
        "variance": variance_score(reference_data, window)
    }




def calibrate_thresholds(
    rng, reference_data, reference_mean, reference_covariance
):
    """Calibrate each detector using no-drift windows."""

    euclidean_scores = []
    mahalanobis_scores = []
    variance_scores = []

    for _ in range(CALIBRATION_WINDOWS):
        window = rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )

        scores = calculate_scores(
            reference_mean,
            reference_covariance,
            reference_data,
            window
        )

        euclidean_scores.append(scores["euclidean"])
        mahalanobis_scores.append(scores["mahalanobis"])
        variance_scores.append(scores["variance"])

    thresholds = {
        "euclidean": np.quantile(
            euclidean_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "mahalanobis": np.quantile(
            mahalanobis_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "variance": np.quantile(
            variance_scores, 1 - TARGET_FALSE_ALARM_RATE
        )
    }

    return thresholds



print("Data-generation and calibration functions defined successfully.")


Data-generation and calibration functions defined successfully.


In [ ]:

# Pilot test: one seed only
pilot_rng = np.random.default_rng(2026)

(
    pilot_reference,
    pilot_mean,
    pilot_covariance,
    pilot_variance
) = generate_reference(pilot_rng)

pilot_thresholds = calibrate_thresholds(
    pilot_rng,
    pilot_mean,
    pilot_covariance,
    pilot_variance
)

pilot_window = generate_window(pilot_rng)

pilot_scores = calculate_scores(
    pilot_mean,
    pilot_covariance,
    pilot_variance,
    pilot_window
)

print("Pilot test completed.")
print("\nCalibrated thresholds:")
for detector, threshold in pilot_thresholds.items():
    print(f"{detector}: {threshold:.6f}")

print("\nExample no-drift scores:")
for detector, score in pilot_scores.items():
    print(f"{detector}: {score:.6f}")

print("\nPilot checks:")
print("Reference shape:", pilot_reference.shape)
print("All thresholds finite:", all(np.isfinite(v) for v in pilot_thresholds.values()))
print("All scores finite:", all(np.isfinite(v) for v in pilot_scores.values()))


Pilot test completed.

Calibrated thresholds:
euclidean: 1.263540
mahalanobis: 1.271008
variance: 1.183811

Example no-drift scores:
euclidean: 0.689658
mahalanobis: 0.690582
variance: 0.934469

Pilot checks:
Reference shape: (500, 16)
All thresholds finite: True
All scores finite: True


In [ ]:

# Independent false-alarm validation for pilot seed 2026
validation_rng = np.random.default_rng(902026)

false_alarms = {
    "euclidean": 0,
    "mahalanobis": 0,
    "variance": 0
}

for _ in range(VALIDATION_WINDOWS):
    window = validation_rng.normal(
        loc=pilot_mean,
        scale=1.0,
        size=(WINDOW_SIZE, EMBEDDING_DIM)
    )

    scores = calculate_scores(
        pilot_mean,
        pilot_covariance,
        pilot_variance,
        window
    )

    for detector, score in scores.items():
        if score > pilot_thresholds[detector]:
            false_alarms[detector] += 1

print("Independent no-drift validation (seed 2026)")
for detector, count in false_alarms.items():
    rate = count / VALIDATION_WINDOWS
    print(
        f"{detector}: {count}/{VALIDATION_WINDOWS} "
        f"= {rate:.2%} false-alarm rate"
    )

print("\nTarget false-alarm rate: 1.00%")


Independent no-drift validation (seed 2026)
euclidean: 62/5000 = 1.24% false-alarm rate
mahalanobis: 72/5000 = 1.44% false-alarm rate
variance: 44/5000 = 0.88% false-alarm rate

Target false-alarm rate: 1.00%


In [ ]:

# Verify the pilot configuration and record its provenance
pilot_record = {
    "experiment_type": "reconstructed_synthetic_gaussian",
    "seed": 2026,
    "embedding_dim": EMBEDDING_DIM,
    "reference_samples": REFERENCE_SAMPLES,
    "calibration_windows": CALIBRATION_WINDOWS,
    "validation_windows": VALIDATION_WINDOWS,
    "window_size": WINDOW_SIZE,
    "target_false_alarm_rate": TARGET_FALSE_ALARM_RATE,
    "detectors": list(pilot_thresholds.keys()),
    "validation_rates": {
        detector: count / VALIDATION_WINDOWS
        for detector, count in false_alarms.items()
    }
}

print(pd.Series(pilot_record).to_string())


experiment_type                             reconstructed_synthetic_gaussian
seed                                                                    2026
embedding_dim                                                             16
reference_samples                                                        500
calibration_windows                                                     2000
validation_windows                                                      5000
window_size                                                               20
target_false_alarm_rate                                                 0.01
detectors                                 [euclidean, mahalanobis, variance]
validation_rates           {'euclidean': 0.0124, 'mahalanobis': 0.0144, '...


In [ ]:

# Check that the reconstructed experiment configuration
# matches the documented report settings.

expected_config = {
    "seeds": list(range(2026, 2036)),
    "embedding_dim": 16,
    "reference_samples": 500,
    "calibration_windows": 2000,
    "test_windows": 1000,
    "validation_windows": 5000,
    "window_size": 20,
    "mean_shift_levels": [0.00, 0.02, 0.05, 0.10, 0.15],
    "variance_scale_levels": [1.00, 1.10, 1.25, 1.50, 2.00],
}

print("Documented configuration:")
for key, value in expected_config.items():
    print(f"{key}: {value}")


Documented configuration:
seeds: [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035]
embedding_dim: 16
reference_samples: 500
calibration_windows: 2000
test_windows: 1000
validation_windows: 5000
window_size: 20
mean_shift_levels: [0.0, 0.02, 0.05, 0.1, 0.15]
variance_scale_levels: [1.0, 1.1, 1.25, 1.5, 2.0]


In [ ]:

# Sanity test: variance detector response to increased spread
sanity_rng = np.random.default_rng(777)

normal_window = sanity_rng.normal(
    loc=0.0, scale=1.0,
    size=(WINDOW_SIZE, EMBEDDING_DIM)
)

high_variance_window = sanity_rng.normal(
    loc=0.0, scale=np.sqrt(2.0),
    size=(WINDOW_SIZE, EMBEDDING_DIM)
)

normal_score = variance_score(pilot_variance, normal_window)
high_variance_score = variance_score(pilot_variance, high_variance_window)

print(f"Normal-spread score: {normal_score:.4f}")
print(f"High-spread score:   {high_variance_score:.4f}")
print("High-spread score is larger:", high_variance_score > normal_score)


Normal-spread score: 0.9489
High-spread score:   2.0060
High-spread score is larger: True


In [ ]:

def run_one_seed(seed):
    rng = np.random.default_rng(seed)

    (
        reference,
        reference_mean,
        reference_covariance,
        reference_variance
    ) = generate_reference(rng)

    thresholds = calibrate_thresholds(
        rng,
        reference,
        reference_mean,
        reference_covariance,
    )

    results = []

    scenarios = [
        ("mean_drift", level, level, 1.0)
        for level in MEAN_SHIFT_LEVELS
    ] + [
        ("variance_drift", level, 0.0, level)
        for level in VARIANCE_SCALE_LEVELS
    ]

    for scenario, level, mean_shift, variance_scale in scenarios:
        detections = {
            "euclidean": 0,
            "mahalanobis": 0,
            "variance": 0
        }

        for _ in range(TEST_WINDOWS):
            window = generate_window(
                rng,
                mean_shift=mean_shift,
                variance_scale=variance_scale
            )

            scores = calculate_scores(
                reference_mean,
                reference_covariance,
                reference,
                window
            )

            for detector, score in scores.items():
                if score > thresholds[detector]:
                    detections[detector] += 1

        results.append({
            "seed": seed,
            "scenario": scenario,
            "level": level,
            "euclidean_detection_rate": detections["euclidean"] / TEST_WINDOWS,
            "mahalanobis_detection_rate": detections["mahalanobis"] / TEST_WINDOWS,
            "variance_detection_rate": detections["variance"] / TEST_WINDOWS
        })

    return results


print("Full experiment runner defined successfully.")


Full experiment runner defined successfully.


In [ ]:

from pathlib import Path
import json
import time

# All outputs are isolated from the existing corrected CSV files.
OUTPUT_DIR = Path("reconstruction_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

all_results = []
all_validation = []

start_time = time.time()

for seed in SEEDS:
    print(f"Running seed {seed}...")

    # Detection results for this seed
    seed_results = run_one_seed(seed)
    all_results.extend(seed_results)

    # Independent reference and calibration for validation
    validation_rng = np.random.default_rng(seed + 100_000)

    (
        reference,
        reference_mean,
        reference_covariance,
        reference_variance
    ) = generate_reference(validation_rng)

    thresholds = calibrate_thresholds(
        validation_rng,
        reference_mean,
        reference_covariance,
        reference_variance
    )

    false_alarms = {
        detector: 0
        for detector in thresholds
    }

    for _ in range(VALIDATION_WINDOWS):
        window = validation_rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )

        scores = calculate_scores(
            reference_mean,
            reference_covariance,
            reference_variance,
            window
        )

        for detector, score in scores.items():
            if score > thresholds[detector]:
                false_alarms[detector] += 1

    for detector, count in false_alarms.items():
        all_validation.append({
            "seed": seed,
            "detector": detector,
            "validation_windows": VALIDATION_WINDOWS,
            "false_alarm_count": count,
            "false_alarm_rate": count / VALIDATION_WINDOWS
        })

results_df = pd.DataFrame(all_results)
validation_df = pd.DataFrame(all_validation)

# Summarize each scenario and detector across the 10 seeds.
detector_columns = [
    "euclidean_detection_rate",
    "mahalanobis_detection_rate",
    "variance_detection_rate"
]

summary_df = (
    results_df
    .groupby(["scenario", "level"])[detector_columns]
    .agg(["mean", "std"])
)

# Save separate reconstruction files.
results_df.to_csv(
    OUTPUT_DIR / "reconstructed_multiseed_results.csv",
    index=False
)
validation_df.to_csv(
    OUTPUT_DIR / "reconstructed_false_alarm_validation.csv",
    index=False
)
summary_df.to_csv(
    OUTPUT_DIR / "reconstructed_detection_summary.csv"
)

provenance = {
    "status": "reconstructed experiment; not the recovered original source",
    "seeds": list(SEEDS),
    "embedding_dim": EMBEDDING_DIM,
    "reference_samples": REFERENCE_SAMPLES,
    "calibration_windows": CALIBRATION_WINDOWS,
    "test_windows_per_scenario": TEST_WINDOWS,
    "validation_windows_per_seed": VALIDATION_WINDOWS,
    "window_size": WINDOW_SIZE,
    "mean_shift_levels": MEAN_SHIFT_LEVELS,
    "variance_scale_levels": VARIANCE_SCALE_LEVELS,
    "detectors": ["euclidean", "mahalanobis", "variance"],
    "note": (
        "Synthetic Gaussian data only. Validation is conditional on "
        "the generated reference mean; compare methodology before "
        "interpreting results against prior CSVs."
    )
}

with open(OUTPUT_DIR / "provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print("\nExperiment completed.")
print(f"Elapsed time: {(time.time() - start_time) / 60:.1f} minutes")
print("Detection rows:", len(results_df))
print("Validation rows:", len(validation_df))
print("Output directory:", OUTPUT_DIR.resolve())
print("\nMean false-alarm rates:")
print(
    validation_df.groupby("detector")["false_alarm_rate"]
    .mean()
    .map(lambda x: f"{x:.2%}")
)


Running seed 2026...
Running seed 2027...
Running seed 2028...
Running seed 2029...
Running seed 2030...
Running seed 2031...
Running seed 2032...
Running seed 2033...
Running seed 2034...
Running seed 2035...

Experiment completed.
Elapsed time: 1.2 minutes
Detection rows: 100
Validation rows: 30
Output directory: /content/reconstruction_outputs

Mean false-alarm rates:
detector
euclidean      1.06%
mahalanobis    1.09%
variance       1.02%
Name: false_alarm_rate, dtype: object


In [ ]:

import pandas as pd

github_csv_url = (
    "https://raw.githubusercontent.com/12998951/"
    "Real-Time-Embedding-Level-Drift-Detection/"
    "main/results/corrected/corrected_detection_summary.csv"
)

original_summary = pd.read_csv(github_csv_url)

print("Committed summary shape:", original_summary.shape)

print("\nCommitted summary columns:")
print(original_summary.columns.tolist())

print("\nFirst 12 rows of the committed summary:")
display(original_summary.head(12))

print("\nReconstructed summary:")
display(summary_df.reset_index())


Committed summary shape: (10, 8)

Committed summary columns:
['scenario', 'level', 'euclidean_detection_rate_mean', 'euclidean_detection_rate_std', 'mahalanobis_detection_rate_mean', 'mahalanobis_detection_rate_std', 'variance_detection_rate_mean', 'variance_detection_rate_std']

First 12 rows of the committed summary:


,scenario,level,euclidean_detection_rate_mean,euclidean_detection_rate_std,mahalanobis_detection_rate_mean,mahalanobis_detection_rate_std,variance_detection_rate_mean,variance_detection_rate_std
0,mean_drift,0.00,0.0120,0.004243,0.0105,0.004767,0.0099,0.005486
1,mean_drift,0.02,0.0123,0.005697,0.0122,0.006861,0.0086,0.004088
2,mean_drift,0.05,0.0175,0.004378,0.0144,0.004033,0.0083,0.002908
3,mean_drift,0.10,0.0429,0.011571,0.0389,0.015466,0.0091,0.004771
4,mean_drift,0.15,0.1336,0.017102,0.1138,0.018890,0.0101,0.003446
5,variance_drift,1.00,0.0112,0.004984,0.0111,0.003929,0.0097,0.003164
6,variance_drift,1.10,0.0473,0.006913,0.0444,0.007763,0.5383,0.028879
7,variance_drift,1.25,0.2113,0.037443,0.1943,0.036896,0.9992,0.001317
8,variance_drift,1.50,0.5798,0.028452,0.5585,0.030211,1.0000,0.000000
9,variance_drift,2.00,0.9509,0.007852,0.9427,0.009719,1.0000,0.000000



Reconstructed summary:


scenario level euclidean_detection_rate            \
                                            mean       std   
0      mean_drift  0.00                   0.0144  0.007457   
1      mean_drift  0.02                   0.0155  0.007184   
2      mean_drift  0.05                   0.0239  0.010744   
3      mean_drift  0.10                   0.0614  0.024015   
4      mean_drift  0.15                   0.1605  0.050474   
5  variance_drift  1.00                   0.0104  0.005929   
6  variance_drift  1.10                   0.0288  0.009612   
7  variance_drift  1.25                   0.0694  0.020882   
8  variance_drift  1.50                   0.1799  0.035629   
9  variance_drift  2.00                   0.4566  0.031415   

  mahalanobis_detection_rate           variance_detection_rate            
                        mean       std                    mean       std  
0                     0.0145  0.007322                  0.0089  0.002923  
1                     0.0169  0.006707                  0.0101  0.002025  
2                     0.0244  0.011549                  0.0101  0.003929  
3                     0.0579  0.022733                  0.0092  0.003360  
4                     0.1505  0.046819                  0.0105  0.004353  
5                     0.0113  0.005755                  0.0082  0.003084  
6                     0.0289  0.008089                  0.1406  0.017206  
7                     0.0702  0.018546                  0.6979  0.026232  
8                     0.1821  0.035899                  0.9959  0.002807  
9                     0.4499  0.026320                  1.0000  0.000000

In [ ]:

import inspect

print("Current variance detector implementation:\n")
print(inspect.getsource(variance_score))

print("\nCurrent calibration implementation:\n")
print(inspect.getsource(calibrate_thresholds))


Current variance detector implementation:

def variance_score(reference_variance, window):
    """Ratio of current-window spread to reference spread."""
    current_variance = np.mean(
        np.var(window, axis=0, ddof=1)
    )

    reference_spread = np.mean(reference_variance)

    if reference_spread <= 0:
        raise ValueError("Reference variance must be positive.")

    return float(current_variance / reference_spread)


Current calibration implementation:

def calibrate_thresholds(rng, reference_mean,
                         reference_covariance, reference_variance):
    """Calibrate each detector at the 99th percentile."""
    calibration_scores = {
        "euclidean": [],
        "mahalanobis": [],
        "variance": []
    }

    for _ in range(CALIBRATION_WINDOWS):
        window = rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )

        scores = calculate_scores(
            reference_mean,
  

In [ ]:

# Diagnostic comparison for seed 2026
diagnostic_rng = np.random.default_rng(2026)

(
    diagnostic_reference,
    diagnostic_mean,
    diagnostic_covariance,
    diagnostic_variance
) = generate_reference(diagnostic_rng)

diagnostic_thresholds = calibrate_thresholds(
    diagnostic_rng,
    diagnostic_mean,
    diagnostic_covariance,
    diagnostic_variance
)

print("Seed 2026 thresholds:")
for name, threshold in diagnostic_thresholds.items():
    print(f"{name}: {threshold:.6f}")

print("\nVariance detector diagnostic:")
print("Variance scale | Mean score | Detection rate")

for scale in [1.0, 1.1, 1.25, 1.5, 2.0]:
    scores = []

    for _ in range(1000):
        window = generate_window(
            diagnostic_rng,
            mean_shift=0.0,
            variance_scale=scale
        )
        scores.append(
            variance_score(diagnostic_variance, window)
        )

    detection_rate = np.mean(
        np.array(scores) > diagnostic_thresholds["variance"]
    )

    print(
        f"{scale:13.2f} | {np.mean(scores):10.4f} | "
        f"{detection_rate:14.2%}"
    )


Seed 2026 thresholds:
euclidean: 1.263540
mahalanobis: 1.271008
variance: 1.183811

Variance detector diagnostic:
Variance scale | Mean score | Detection rate
         1.00 |     0.9829 |          0.40%
         1.10 |     1.0798 |         12.30%
         1.25 |     1.2254 |         65.60%
         1.50 |     1.4720 |         99.50%
         2.00 |     1.9578 |        100.00%


In [ ]:

# Inspect the variance-score calibration distribution for seed 2026
calibration_rng = np.random.default_rng(2026)

(
    ref,
    ref_mean,
    ref_cov,
    ref_var
) = generate_reference(calibration_rng)

variance_calibration_scores = []

for _ in range(CALIBRATION_WINDOWS):
    window = calibration_rng.normal(
        loc=ref_mean,
        scale=1.0,
        size=(WINDOW_SIZE, EMBEDDING_DIM)
    )
    variance_calibration_scores.append(
        variance_score(ref_var, window)
    )

variance_calibration_scores = np.array(variance_calibration_scores)

print("Calibration score statistics:")
print(f"Mean:   {variance_calibration_scores.mean():.6f}")
print(f"Median: {np.median(variance_calibration_scores):.6f}")
print(f"95th percentile: {np.quantile(variance_calibration_scores, 0.95):.6f}")
print(f"99th percentile: {np.quantile(variance_calibration_scores, 0.99):.6f}")
print(f"99.5th percentile: {np.quantile(variance_calibration_scores, 0.995):.6f}")


Calibration score statistics:
Mean:   0.981189
Median: 0.977593
95th percentile: 1.118594
99th percentile: 1.183811
99.5th percentile: 1.215227


In [ ]:

# Step 17: Test the updated variance detector and calibration

test_rng = np.random.default_rng(2026)

test_reference, test_mean, test_covariance, test_variance = generate_reference(test_rng)

test_window = generate_window(test_rng)

test_scores = calculate_scores(
    test_mean,
    test_covariance,
    test_reference,
    test_window
)

print("Test scores:", test_scores)

test_thresholds = calibrate_thresholds(
    test_rng,
    test_reference,
    test_mean,
    test_covariance
)

print("Calibration thresholds:", test_thresholds)


Test scores: {'euclidean': np.float64(0.8795551180967269), 'mahalanobis': 0.8447800757715748, 'variance': 0.9449166611583724}
Calibration thresholds: {'euclidean': np.float64(1.263539506659525), 'mahalanobis': np.float64(1.271007971160222), 'variance': np.float64(1.1268745224647703)}


In [ ]:

# Step 20: Test the corrected runner on one seed
pilot_results = run_one_seed(2026)

pilot_df = pd.DataFrame(pilot_results)

print("Pilot experiment completed.")
display(pilot_df.round(4))


Pilot experiment completed.


,seed,scenario,level,euclidean_detection_rate,mahalanobis_detection_rate,variance_detection_rate
0,2026,mean_drift,0.00,0.011,0.014,0.004
1,2026,mean_drift,0.02,0.015,0.019,0.007
2,2026,mean_drift,0.05,0.028,0.029,0.008
3,2026,mean_drift,0.10,0.043,0.042,0.005
4,2026,mean_drift,0.15,0.173,0.166,0.004
5,2026,variance_drift,1.00,0.008,0.012,0.007
6,2026,variance_drift,1.10,0.029,0.032,0.124
7,2026,variance_drift,1.25,0.074,0.076,0.689
8,2026,variance_drift,1.50,0.205,0.213,0.996
9,2026,variance_drift,2.00,0.454,0.466,1.000


In [ ]:

# Step 21: Full reconstructed experiment
all_results = []

for seed in SEEDS:
    all_results.extend(run_one_seed(seed))

all_results_df = pd.DataFrame(all_results)

print("Full reconstructed experiment completed.")
print("Total result rows:", len(all_results_df))

display(
    all_results_df.groupby(["scenario", "level"])[
        [
            "euclidean_detection_rate",
            "mahalanobis_detection_rate",
            "variance_detection_rate"
        ]
    ].agg(["mean", "std"]).round(4)
)


Full reconstructed experiment completed.
Total result rows: 100


euclidean_detection_rate          \
                                         mean     std   
scenario       level                                    
mean_drift     0.00                    0.0144  0.0075   
               0.02                    0.0155  0.0072   
               0.05                    0.0239  0.0107   
               0.10                    0.0614  0.0240   
               0.15                    0.1605  0.0505   
variance_drift 1.00                    0.0104  0.0059   
               1.10                    0.0288  0.0096   
               1.25                    0.0694  0.0209   
               1.50                    0.1799  0.0356   
               2.00                    0.4566  0.0314   

                     mahalanobis_detection_rate          \
                                           mean     std   
scenario       level                                      
mean_drift     0.00                      0.0145  0.0073   
               0.02                      0.0169  0.0067   
               0.05                      0.0244  0.0115   
               0.10                      0.0579  0.0227   
               0.15                      0.1505  0.0468   
variance_drift 1.00                      0.0113  0.0058   
               1.10                      0.0289  0.0081   
               1.25                      0.0702  0.0185   
               1.50                      0.1821  0.0359   
               2.00                      0.4499  0.0263   

                     variance_detection_rate          
                                        mean     std  
scenario       level                                  
mean_drift     0.00                   0.0089  0.0029  
               0.02                   0.0101  0.0020  
               0.05                   0.0101  0.0039  
               0.10                   0.0092  0.0034  
               0.15                   0.0105  0.0044  
variance_drift 1.00                   0.0082  0.0031  
               1.10                   0.1406  0.0172  
               1.25                   0.6979  0.0262  
               1.50                   0.9959  0.0028  
               2.00                   1.0000  0.0000

In [ ]:

# Step 22: Inspect the variance calibration for seed 2026

diagnostic_rng = np.random.default_rng(2026)

(
    diagnostic_reference,
    diagnostic_mean,
    diagnostic_covariance,
    diagnostic_variance
) = generate_reference(diagnostic_rng)

diagnostic_thresholds = calibrate_thresholds(
    diagnostic_rng,
    diagnostic_reference,
    diagnostic_mean,
    diagnostic_covariance
)

print("Seed:", 2026)
print("Reference spread:", np.mean(
    np.sum(
        (diagnostic_reference - diagnostic_reference.mean(axis=0)) ** 2,
        axis=1
    )
))
print("Variance threshold:", diagnostic_thresholds["variance"])


Seed: 2026
Reference spread: 16.264778622616276
Variance threshold: 1.1268745224647703


In [ ]:
# Step 23: Inspect calibration settings
print("SEEDS:", list(SEEDS))
print("CALIBRATION_WINDOWS:", CALIBRATION_WINDOWS)
print("TEST_WINDOWS:", TEST_WINDOWS)
print("WINDOW_SIZE:", WINDOW_SIZE)
print("EMBEDDING_DIM:", EMBEDDING_DIM)
print("TARGET_FALSE_ALARM_RATE:", TARGET_FALSE_ALARM_RATE)

SEEDS: [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035]
CALIBRATION_WINDOWS: 2000
TEST_WINDOWS: 1000
WINDOW_SIZE: 20
EMBEDDING_DIM: 16
TARGET_FALSE_ALARM_RATE: 0.01


In [ ]:
# Step 25: Inspect the corrected evaluation setup

print("Reference samples:", REFERENCE_SAMPLES)
print("Calibration windows:", CALIBRATION_WINDOWS)
print("Test windows:", TEST_WINDOWS)
print("Window size:", WINDOW_SIZE)
print("Embedding dimensions:", EMBEDDING_DIM)
print("False-alarm target:", TARGET_FALSE_ALARM_RATE)

print("\nVariance score function:")
import inspect
print(inspect.getsource(variance_score))

Reference samples: 500
Calibration windows: 2000
Test windows: 1000
Window size: 20
Embedding dimensions: 16
False-alarm target: 0.01

Variance score function:
def variance_score(reference_data, current_window):
    """Ratio of current-window spread to reference-data spread."""

    reference_spread = np.mean(
        np.sum(
            (reference_data - reference_data.mean(axis=0)) ** 2,
            axis=1
        )
    )

    current_spread = np.mean(
        np.sum(
            (current_window - current_window.mean(axis=0)) ** 2,
            axis=1
        )
    )

    return float(current_spread / reference_spread)



In [ ]:
# Step 26: Inspect the current reconstruction functions
import inspect

for name in ["calibrate_thresholds", "generate_window", "run_one_seed"]:
    print(f"\n--- {name} ---")
    obj = globals().get(name)

    if obj is None:
        print("Not defined in this notebook.")
    else:
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Source unavailable; function exists.")


--- calibrate_thresholds ---
def calibrate_thresholds(
    rng, reference_data, reference_mean, reference_covariance
):
    """Calibrate each detector using no-drift windows."""

    euclidean_scores = []
    mahalanobis_scores = []
    variance_scores = []

    for _ in range(CALIBRATION_WINDOWS):
        window = rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )

        scores = calculate_scores(
            reference_mean,
            reference_covariance,
            reference_data,
            window
        )

        euclidean_scores.append(scores["euclidean"])
        mahalanobis_scores.append(scores["mahalanobis"])
        variance_scores.append(scores["variance"])

    thresholds = {
        "euclidean": np.quantile(
            euclidean_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "mahalanobis": np.quantile(
            mahalanobis_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "var

In [ ]:
def run_one_seed(seed):
    # 1. Reference data
    ref_rng = np.random.default_rng(seed)
    reference = ref_rng.normal(
        0, 1, size=(REFERENCE_SAMPLES, EMBEDDING_DIM)
    )
    reference_mean = reference.mean(axis=0)
    reference_covariance = np.cov(reference, rowvar=False)

    # 2. Independent calibration data
    cal_rng = np.random.default_rng(seed + 10000)
    euclidean_scores = []
    mahalanobis_scores = []
    variance_scores = []

    for _ in range(CALIBRATION_WINDOWS):
        window = cal_rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )
        scores = calculate_scores(
            reference_mean, reference_covariance, reference, window
        )
        euclidean_scores.append(scores["euclidean"])
        mahalanobis_scores.append(scores["mahalanobis"])
        variance_scores.append(scores["variance"])

    thresholds = {
        "euclidean": np.quantile(
            euclidean_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "mahalanobis": np.quantile(
            mahalanobis_scores, 1 - TARGET_FALSE_ALARM_RATE
        ),
        "variance": np.quantile(
            variance_scores, 1 - TARGET_FALSE_ALARM_RATE
        )
    }

    # 3. Independent test data
    test_rng = np.random.default_rng(seed + 20000)
    results = []

    scenarios = (
        [("mean_drift", level) for level in MEAN_SHIFT_LEVELS]
        + [("variance_drift", level) for level in VARIANCE_SCALE_LEVELS]
    )

    for scenario, level in scenarios:
        detections = {
            "euclidean": 0,
            "mahalanobis": 0,
            "variance": 0
        }

        for _ in range(TEST_WINDOWS):
            if scenario == "mean_drift":
                window_mean = reference_mean + level
                scale = 1.0
            else:
                window_mean = reference_mean
                scale = level

            window = test_rng.normal(
                loc=window_mean,
                scale=scale,
                size=(WINDOW_SIZE, EMBEDDING_DIM)
            )

            scores = calculate_scores(
                reference_mean, reference_covariance, reference, window
            )

            for detector, score in scores.items():
                if score > thresholds[detector]:
                    detections[detector] += 1

        results.append({
            "seed": seed,
            "scenario": scenario,
            "level": level,
            "euclidean_detection_rate":
                detections["euclidean"] / TEST_WINDOWS,
            "mahalanobis_detection_rate":
                detections["mahalanobis"] / TEST_WINDOWS,
            "variance_detection_rate":
                detections["variance"] / TEST_WINDOWS
        })

    return results

In [ ]:
print(run_one_seed)

<function run_one_seed at 0x7a459d2bdf80>


In [ ]:
import inspect
print(inspect.getsource(calculate_scores))

def calculate_scores(reference_mean, reference_covariance,
                     reference_data, window):
    """Calculate all three detector scores for a window."""
    return {
        "euclidean": euclidean_score(reference_mean, window),
        "mahalanobis": mahalanobis_score(
            reference_mean, reference_covariance, window
        ),
        "variance": variance_score(reference_data, window)
    }



In [ ]:
import inspect

print("=== EUCLIDEAN SCORE ===")
print(inspect.getsource(euclidean_score))

print("\n=== MAHALANOBIS SCORE ===")
print(inspect.getsource(mahalanobis_score))

print("\n=== VARIANCE SCORE ===")
print(inspect.getsource(variance_score))

=== EUCLIDEAN SCORE ===
def euclidean_score(reference_mean, window):
    """Distance between reference mean and current-window mean."""
    window_mean = np.mean(window, axis=0)
    return np.linalg.norm(window_mean - reference_mean)


=== MAHALANOBIS SCORE ===
def mahalanobis_score(reference_mean, reference_covariance, window):
    """Covariance-normalized distance between window and reference means."""
    window_mean = np.mean(window, axis=0)
    delta = window_mean - reference_mean

    regularized_covariance = (
        reference_covariance
        + 1e-6 * np.eye(len(reference_mean))
    )

    inverse_covariance = np.linalg.pinv(regularized_covariance)
    squared_distance = delta @ inverse_covariance @ delta

    return float(np.sqrt(max(squared_distance, 0.0)))


=== VARIANCE SCORE ===
def variance_score(reference_data, current_window):
    """Ratio of current-window spread to reference-data spread."""

    reference_spread = np.mean(
        np.sum(
            (reference_dat

In [ ]:
pilot_results = run_one_seed(2026)

import pandas as pd
pilot_df = pd.DataFrame(pilot_results)

print(pilot_df.to_string(index=False))

 seed       scenario  level  euclidean_detection_rate  mahalanobis_detection_rate  variance_detection_rate
 2026     mean_drift   0.00                     0.016                       0.013                    0.004
 2026     mean_drift   0.02                     0.023                       0.027                    0.005
 2026     mean_drift   0.05                     0.016                       0.018                    0.008
 2026     mean_drift   0.10                     0.067                       0.070                    0.008
 2026     mean_drift   0.15                     0.148                       0.135                    0.009
 2026 variance_drift   1.00                     0.014                       0.017                    0.006
 2026 variance_drift   1.10                     0.058                       0.062                    0.512
 2026 variance_drift   1.25                     0.280                       0.267                    0.998
 2026 variance_drift   1.50          

In [ ]:
# Step 31: Full corrected multi-seed evaluation

all_results = []

for seed in range(2026, 2036):
    all_results.extend(run_one_seed(seed))
    print(f"Completed seed {seed}")

results_df = pd.DataFrame(all_results)

summary_df = (
    results_df
    .groupby(["scenario", "level"])[
        [
            "euclidean_detection_rate",
            "mahalanobis_detection_rate",
            "variance_detection_rate"
        ]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("\nFull experiment results:")
display(summary_df)

Completed seed 2026
Completed seed 2027
Completed seed 2028
Completed seed 2029
Completed seed 2030
Completed seed 2031
Completed seed 2032
Completed seed 2033
Completed seed 2034
Completed seed 2035

Full experiment results:


euclidean_detection_rate          \
                                         mean     std   
scenario       level                                    
mean_drift     0.00                    0.0120  0.0042   
               0.02                    0.0123  0.0057   
               0.05                    0.0175  0.0044   
               0.10                    0.0429  0.0116   
               0.15                    0.1336  0.0171   
variance_drift 1.00                    0.0112  0.0050   
               1.10                    0.0473  0.0069   
               1.25                    0.2113  0.0374   
               1.50                    0.5798  0.0285   
               2.00                    0.9509  0.0079   

                     mahalanobis_detection_rate          \
                                           mean     std   
scenario       level                                      
mean_drift     0.00                      0.0105  0.0048   
               0.02                      0.0122  0.0069   
               0.05                      0.0144  0.0040   
               0.10                      0.0389  0.0155   
               0.15                      0.1138  0.0189   
variance_drift 1.00                      0.0111  0.0039   
               1.10                      0.0444  0.0078   
               1.25                      0.1943  0.0369   
               1.50                      0.5585  0.0302   
               2.00                      0.9427  0.0097   

                     variance_detection_rate          
                                        mean     std  
scenario       level                                  
mean_drift     0.00                   0.0099  0.0055  
               0.02                   0.0086  0.0041  
               0.05                   0.0083  0.0029  
               0.10                   0.0091  0.0048  
               0.15                   0.0101  0.0034  
variance_drift 1.00                   0.0097  0.0032  
               1.10                   0.5383  0.0289  
               1.25                   0.9992  0.0013  
               1.50                   1.0000  0.0000  
               2.00                   1.0000  0.0000

In [ ]:
# Step 32: Final integrity checks

print("Total result rows:", len(results_df))
print("Unique seeds:", results_df["seed"].nunique())
print("Unique scenarios:", results_df[["scenario", "level"]].drop_duplicates().shape[0])
print("Missing values:")
print(results_df.isna().sum())

assert len(results_df) == 100, "Expected 100 result rows"
assert results_df["seed"].nunique() == 10, "Expected 10 seeds"
assert results_df[["scenario", "level"]].drop_duplicates().shape[0] == 10, \
    "Expected 10 unique scenarios"

print("\nAll integrity checks passed.")

Total result rows: 100
Unique seeds: 10
Unique scenarios: 10
Missing values:
seed                          0
scenario                      0
level                         0
euclidean_detection_rate      0
mahalanobis_detection_rate    0
variance_detection_rate       0
dtype: int64

All integrity checks passed.


In [ ]:
# Step 33: Save reconstructed experiment outputs

results_df.to_csv(
    "reconstructed_multiseed_detection_results.csv",
    index=False
)

summary_export = (
    results_df
    .groupby(["scenario", "level"])[
        [
            "euclidean_detection_rate",
            "mahalanobis_detection_rate",
            "variance_detection_rate"
        ]
    ]
    .agg(["mean", "std"])
)

summary_export.to_csv("reconstructed_detection_summary.csv")

print("Saved reconstructed results and summary.")

Saved reconstructed results and summary.


In [ ]:
# Step 34: Independent false-alarm validation

VALIDATION_WINDOWS = 5000
validation_results = []

for seed in range(2026, 2036):
    # Recreate reference data
    ref_rng = np.random.default_rng(seed)
    reference = ref_rng.normal(
        0, 1, size=(REFERENCE_SAMPLES, EMBEDDING_DIM)
    )
    reference_mean = reference.mean(axis=0)
    reference_covariance = np.cov(reference, rowvar=False)

    # Recreate calibration thresholds
    cal_rng = np.random.default_rng(seed + 10000)
    euclidean_scores = []
    mahalanobis_scores = []
    variance_scores = []

    for _ in range(CALIBRATION_WINDOWS):
        window = cal_rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )
        scores = calculate_scores(
            reference_mean, reference_covariance, reference, window
        )
        euclidean_scores.append(scores["euclidean"])
        mahalanobis_scores.append(scores["mahalanobis"])
        variance_scores.append(scores["variance"])

    thresholds = {
        "euclidean": np.quantile(euclidean_scores, 0.99),
        "mahalanobis": np.quantile(mahalanobis_scores, 0.99),
        "variance": np.quantile(variance_scores, 0.99)
    }

    # Independent validation stream
    val_rng = np.random.default_rng(seed + 30000)
    false_alarms = {
        "euclidean": 0,
        "mahalanobis": 0,
        "variance": 0
    }

    for _ in range(VALIDATION_WINDOWS):
        window = val_rng.normal(
            loc=reference_mean,
            scale=1.0,
            size=(WINDOW_SIZE, EMBEDDING_DIM)
        )
        scores = calculate_scores(
            reference_mean, reference_covariance, reference, window
        )

        for detector, score in scores.items():
            if score > thresholds[detector]:
                false_alarms[detector] += 1

    validation_results.append({
        "seed": seed,
        "euclidean_false_alarm_rate":
            false_alarms["euclidean"] / VALIDATION_WINDOWS,
        "mahalanobis_false_alarm_rate":
            false_alarms["mahalanobis"] / VALIDATION_WINDOWS,
        "variance_false_alarm_rate":
            false_alarms["variance"] / VALIDATION_WINDOWS
    })

validation_df = pd.DataFrame(validation_results)

print("Mean independent false-alarm rates:")
display(
    validation_df.drop(columns="seed").mean().to_frame("mean_rate").round(4)
)

Mean independent false-alarm rates:


,mean_rate
euclidean_false_alarm_rate,0.0110
mahalanobis_false_alarm_rate,0.0101
variance_false_alarm_rate,0.0110


In [ ]:
# Step 35: Save independent false-alarm validation

validation_df.to_csv(
    "reconstructed_independent_false_alarm_validation.csv",
    index=False
)

print("Saved independent false-alarm validation results.")

print("\nPer-seed validation:")
display(validation_df.round(4))

Saved independent false-alarm validation results.

Per-seed validation:


,seed,euclidean_false_alarm_rate,mahalanobis_false_alarm_rate,variance_false_alarm_rate
0,2026,0.0166,0.0166,0.0084
1,2027,0.0100,0.0092,0.0140
2,2028,0.0144,0.0132,0.0092
3,2029,0.0116,0.0112,0.0108
4,2030,0.0056,0.0072,0.0134
5,2031,0.0112,0.0082,0.0088
6,2032,0.0102,0.0094,0.0084
7,2033,0.0090,0.0084,0.0130
8,2034,0.0084,0.0072,0.0094
9,2035,0.0126,0.0102,0.0144


In [ ]:
import os

files_to_check = [
    "corrected_multiseed_detection_results.csv",
    "corrected_detection_summary.csv",
    "independent_false_alarm_validation.csv",
    "reconstructed_multiseed_detection_results.csv",
    "reconstructed_detection_summary.csv",
    "reconstructed_independent_false_alarm_validation.csv"
]

print("Current directory:", os.getcwd())

for filename in files_to_check:
    print(
        f"{filename}: "
        f"{'FOUND' if os.path.exists(filename) else 'NOT FOUND'}"
    )

Current directory: /content
corrected_multiseed_detection_results.csv: FOUND
corrected_detection_summary.csv: FOUND
independent_false_alarm_validation.csv: FOUND
reconstructed_multiseed_detection_results.csv: FOUND
reconstructed_detection_summary.csv: FOUND
reconstructed_independent_false_alarm_validation.csv: FOUND


In [ ]:
import pandas as pd
import numpy as np

# Load original and reconstructed multi-seed results
original = pd.read_csv("corrected_multiseed_detection_results.csv")
reconstructed = pd.read_csv("reconstructed_multiseed_detection_results.csv")

print("=== FILE SHAPES ===")
print("Original:", original.shape)
print("Reconstructed:", reconstructed.shape)

# Compare row-by-row after sorting by scenario and seed
keys = ["seed", "scenario", "level"]
metrics = [
    "euclidean_detection_rate",
    "mahalanobis_detection_rate",
    "variance_detection_rate"
]

original = original.sort_values(keys).reset_index(drop=True)
reconstructed = reconstructed.sort_values(keys).reset_index(drop=True)

print("\n=== SCENARIO/SEED MATCH ===")
print("Same scenario and seed rows:",
      original[keys].equals(reconstructed[keys]))

print("\n=== DETECTION RATE COMPARISON ===")
for metric in metrics:
    if len(original) == len(reconstructed):
        a = original[metric].to_numpy()
        b = reconstructed[metric].to_numpy()
        print(f"\n{metric}")
        print("Exact match:", np.array_equal(a, b))
        print("Maximum absolute difference:", round(float(np.max(np.abs(a-b))), 6))
        print("Mean absolute difference:", round(float(np.mean(np.abs(a-b))), 6))
    else:
        print(f"{metric}: row counts differ; cannot compare row-by-row.")

print("\n=== FIRST 10 ROWS: ORIGINAL VS RECONSTRUCTED ===")
comparison = original[keys + metrics].merge(
    reconstructed[keys + metrics],
    on=keys,
    how="outer",
    suffixes=("_original", "_reconstructed"),
    indicator=True
)
display(comparison.head(10))

=== FILE SHAPES ===
Original: (100, 6)
Reconstructed: (100, 6)

=== SCENARIO/SEED MATCH ===
Same scenario and seed rows: True

=== DETECTION RATE COMPARISON ===

euclidean_detection_rate
Exact match: True
Maximum absolute difference: 0.0
Mean absolute difference: 0.0

mahalanobis_detection_rate
Exact match: True
Maximum absolute difference: 0.0
Mean absolute difference: 0.0

variance_detection_rate
Exact match: True
Maximum absolute difference: 0.0
Mean absolute difference: 0.0

=== FIRST 10 ROWS: ORIGINAL VS RECONSTRUCTED ===


,seed,scenario,level,euclidean_detection_rate_original,mahalanobis_detection_rate_original,variance_detection_rate_original,euclidean_detection_rate_reconstructed,mahalanobis_detection_rate_reconstructed,variance_detection_rate_reconstructed,_merge
0,2026,mean_drift,0.00,0.016,0.013,0.004,0.016,0.013,0.004,both
1,2026,mean_drift,0.02,0.023,0.027,0.005,0.023,0.027,0.005,both
2,2026,mean_drift,0.05,0.016,0.018,0.008,0.016,0.018,0.008,both
3,2026,mean_drift,0.10,0.067,0.070,0.008,0.067,0.070,0.008,both
4,2026,mean_drift,0.15,0.148,0.135,0.009,0.148,0.135,0.009,both
5,2026,variance_drift,1.00,0.014,0.017,0.006,0.014,0.017,0.006,both
6,2026,variance_drift,1.10,0.058,0.062,0.512,0.058,0.062,0.512,both
7,2026,variance_drift,1.25,0.280,0.267,0.998,0.280,0.267,0.998,both
8,2026,variance_drift,1.50,0.607,0.610,1.000,0.607,0.610,1.000,both
9,2026,variance_drift,2.00,0.967,0.959,1.000,0.967,0.959,1.000,both


In [ ]:
import pandas as pd
import numpy as np

# Compare summary CSVs
original_summary = pd.read_csv("corrected_detection_summary.csv")
reconstructed_summary = pd.read_csv("reconstructed_detection_summary.csv")

print("=== SUMMARY FILES ===")
print("Original shape:", original_summary.shape)
print("Reconstructed shape:", reconstructed_summary.shape)
print("\nOriginal columns:", original_summary.columns.tolist())
print("Reconstructed columns:", reconstructed_summary.columns.tolist())

print("\nOriginal summary:")
display(original_summary.head())

print("\nReconstructed summary:")
display(reconstructed_summary.head())

# Compare independent false-alarm validation files
original_validation = pd.read_csv("independent_false_alarm_validation.csv")
reconstructed_validation = pd.read_csv(
    "reconstructed_independent_false_alarm_validation.csv"
)

print("\n=== VALIDATION FILES ===")
print("Original shape:", original_validation.shape)
print("Reconstructed shape:", reconstructed_validation.shape)
print("\nOriginal columns:", original_validation.columns.tolist())
print("Reconstructed columns:", reconstructed_validation.columns.tolist())

print("\nOriginal validation:")
display(original_validation.head())

print("\nReconstructed validation:")
display(reconstructed_validation.head())

=== SUMMARY FILES ===
Original shape: (10, 8)
Reconstructed shape: (12, 8)

Original columns: ['scenario', 'level', 'euclidean_detection_rate_mean', 'euclidean_detection_rate_std', 'mahalanobis_detection_rate_mean', 'mahalanobis_detection_rate_std', 'variance_detection_rate_mean', 'variance_detection_rate_std']
Reconstructed columns: ['Unnamed: 0', 'Unnamed: 1', 'euclidean_detection_rate', 'euclidean_detection_rate.1', 'mahalanobis_detection_rate', 'mahalanobis_detection_rate.1', 'variance_detection_rate', 'variance_detection_rate.1']

Original summary:


,scenario,level,euclidean_detection_rate_mean,euclidean_detection_rate_std,mahalanobis_detection_rate_mean,mahalanobis_detection_rate_std,variance_detection_rate_mean,variance_detection_rate_std
0,mean_drift,0.00,0.0120,0.004243,0.0105,0.004767,0.0099,0.005486
1,mean_drift,0.02,0.0123,0.005697,0.0122,0.006861,0.0086,0.004088
2,mean_drift,0.05,0.0175,0.004378,0.0144,0.004033,0.0083,0.002908
3,mean_drift,0.10,0.0429,0.011571,0.0389,0.015466,0.0091,0.004771
4,mean_drift,0.15,0.1336,0.017102,0.1138,0.018890,0.0101,0.003446



Reconstructed summary:


,Unnamed: 0,Unnamed: 1,euclidean_detection_rate,euclidean_detection_rate.1,mahalanobis_detection_rate,mahalanobis_detection_rate.1,variance_detection_rate,variance_detection_rate.1
0,NaN,NaN,mean,std,mean,std,mean,std
1,scenario,level,NaN,NaN,NaN,NaN,NaN,NaN
2,mean_drift,0.0,0.012,0.004242640687119285,0.010499999999999999,0.004766783215358365,0.0099,0.0054863466897380805
3,mean_drift,0.02,0.0123,0.005696977756280566,0.012199999999999999,0.006860515043833566,0.0086,0.004087922591134905
4,mean_drift,0.05,0.0175,0.004377975178854566,0.0144,0.004033195589934447,0.0083,0.002907843798341919



=== VALIDATION FILES ===
Original shape: (10, 4)
Reconstructed shape: (10, 4)

Original columns: ['seed', 'euclidean_false_alarm_rate', 'mahalanobis_false_alarm_rate', 'variance_false_alarm_rate']
Reconstructed columns: ['seed', 'euclidean_false_alarm_rate', 'mahalanobis_false_alarm_rate', 'variance_false_alarm_rate']

Original validation:


,seed,euclidean_false_alarm_rate,mahalanobis_false_alarm_rate,variance_false_alarm_rate
0,2026,0.0166,0.0166,0.0084
1,2027,0.0100,0.0092,0.0140
2,2028,0.0144,0.0132,0.0092
3,2029,0.0116,0.0112,0.0108
4,2030,0.0056,0.0072,0.0134



Reconstructed validation:


,seed,euclidean_false_alarm_rate,mahalanobis_false_alarm_rate,variance_false_alarm_rate
0,2026,0.0166,0.0166,0.0084
1,2027,0.0100,0.0092,0.0140
2,2028,0.0144,0.0132,0.0092
3,2029,0.0116,0.0112,0.0108
4,2030,0.0056,0.0072,0.0134


In [ ]:
import pandas as pd
import numpy as np

# ---------- 1. Compare summary statistics ----------
original_summary = pd.read_csv("corrected_detection_summary.csv")
reconstructed_summary = pd.read_csv(
    "reconstructed_detection_summary.csv",
    header=[0, 1],
    index_col=[0, 1]
)

# Convert reconstructed multi-level columns to the original format
reconstructed_summary.columns = [
    f"{detector}_{stat}"
    for detector, stat in reconstructed_summary.columns
]

reconstructed_summary = reconstructed_summary.reset_index()
reconstructed_summary.columns = [
    "scenario", "level",
    "euclidean_detection_rate_mean",
    "euclidean_detection_rate_std",
    "mahalanobis_detection_rate_mean",
    "mahalanobis_detection_rate_std",
    "variance_detection_rate_mean",
    "variance_detection_rate_std"
]

original_summary = original_summary.sort_values(
    ["scenario", "level"]
).reset_index(drop=True)

reconstructed_summary = reconstructed_summary.sort_values(
    ["scenario", "level"]
).reset_index(drop=True)

summary_metrics = [
    "euclidean_detection_rate_mean",
    "euclidean_detection_rate_std",
    "mahalanobis_detection_rate_mean",
    "mahalanobis_detection_rate_std",
    "variance_detection_rate_mean",
    "variance_detection_rate_std"
]

print("=== SUMMARY COMPARISON ===")
print("Same scenario/level rows:",
      original_summary[["scenario", "level"]].equals(
          reconstructed_summary[["scenario", "level"]]
      ))

for col in summary_metrics:
    difference = np.abs(
        original_summary[col].to_numpy(dtype=float)
        - reconstructed_summary[col].to_numpy(dtype=float)
    )
    print(
        f"{col}: max difference = {difference.max():.12g}; "
        f"exact match = {np.array_equal(difference, np.zeros_like(difference))}"
    )

# ---------- 2. Compare all validation rows ----------
original_validation = pd.read_csv(
    "independent_false_alarm_validation.csv"
).sort_values("seed").reset_index(drop=True)

reconstructed_validation = pd.read_csv(
    "reconstructed_independent_false_alarm_validation.csv"
).sort_values("seed").reset_index(drop=True)

validation_metrics = [
    "euclidean_false_alarm_rate",
    "mahalanobis_false_alarm_rate",
    "variance_false_alarm_rate"
]

print("\n=== INDEPENDENT VALIDATION COMPARISON ===")
print("Same seed rows:",
      original_validation["seed"].equals(reconstructed_validation["seed"]))

for col in validation_metrics:
    difference = np.abs(
        original_validation[col].to_numpy(dtype=float)
        - reconstructed_validation[col].to_numpy(dtype=float)
    )
    print(
        f"{col}: max difference = {difference.max():.12g}; "
        f"exact match = {np.array_equal(difference, np.zeros_like(difference))}"
    )

print("\nComparison complete. No files were modified.")

=== SUMMARY COMPARISON ===
Same scenario/level rows: True
euclidean_detection_rate_mean: max difference = 0; exact match = True
euclidean_detection_rate_std: max difference = 0; exact match = True
mahalanobis_detection_rate_mean: max difference = 0; exact match = True
mahalanobis_detection_rate_std: max difference = 0; exact match = True
variance_detection_rate_mean: max difference = 0; exact match = True
variance_detection_rate_std: max difference = 0; exact match = True

=== INDEPENDENT VALIDATION COMPARISON ===
Same seed rows: True
euclidean_false_alarm_rate: max difference = 0; exact match = True
mahalanobis_false_alarm_rate: max difference = 0; exact match = True
variance_false_alarm_rate: max difference = 0; exact match = True

Comparison complete. No files were modified.


In [50]:

import os
import zipfile
from google.colab import files

csv_files = [
    "/content/reconstructed_multiseed_detection_results.csv",
    "/content/reconstructed_detection_summary.csv",
    "/content/reconstructed_independent_false_alarm_validation.csv"
]

zip_name = "/content/reconstructed_results.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in csv_files:
        if os.path.exists(file_path):
            z.write(file_path, os.path.basename(file_path))
        else:
            print("File not found:", file_path)

files.download(zip_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>